In [0]:
# ============================================
# GOLD LAYER VALIDATION
# ============================================

from pyspark.sql.functions import *

fact = spark.table("gold.fact_transactions")
dim_client = spark.table("gold.dim_client")
dim_merchant = spark.table("gold.dim_merchant")
dim_date = spark.table("gold.dim_date")

silver_tx = spark.table("silver.transactions_clean")

In [0]:
# Row count validation

fact_count = fact.count()
silver_count = silver_tx.count()

print("Fact rows:", fact_count)
print("Silver rows:", silver_count)
print("Match:", fact_count == silver_count)

Fact rows: 12635227
Silver rows: 12635227
Match: True


In [0]:
# Null surrogate key check

null_checks = fact.select(
    sum(col("date_sk").isNull().cast("int")).alias("null_date_sk"),
    sum(col("client_sk").isNull().cast("int")).alias("null_client_sk"),
    sum(col("merchant_sk").isNull().cast("int")).alias("null_merchant_sk")
)

null_checks.show()

+------------+--------------+----------------+
|null_date_sk|null_client_sk|null_merchant_sk|
+------------+--------------+----------------+
|           0|             0|               0|
+------------+--------------+----------------+



In [0]:
# Duplicate transaction check

duplicates = (
    fact.groupBy("transaction_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate transactions:", duplicates)

Duplicate transactions: 0


In [0]:
# Referential integrity check (merchant)

missing_merchants = (
    fact.join(
        dim_merchant,
        "merchant_sk",
        "left_anti"
    ).count()
)

print("Missing merchant references:", missing_merchants)

Missing merchant references: 0


In [0]:
# SCD2 correctness validation

silver = silver_tx.alias("s")
merchant = dim_merchant.alias("m")

scd2_check = (
    silver
    .withColumn("transaction_date", to_date("date"))
    .join(
        merchant,
        (
            (col("s.merchant_id") == col("m.merchant_id")) &
            (col("transaction_date") >= col("m.valid_from")) &
            (
                (col("transaction_date") <= col("m.valid_to")) |
                col("m.valid_to").isNull()
            )
        ),
        "left"
    )
)

invalid_scd2 = scd2_check.filter(
    col("m.merchant_sk").isNull()
).count()

print("Invalid SCD2 matches:", invalid_scd2)

Invalid SCD2 matches: 0
